<a href="https://colab.research.google.com/github/Thilac01/Statistical-Learning-e22395/blob/main/Statistical_Learning_Assignement7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arjunbhasin2013/ccdata")

print("Path to dataset files:", path)

## Part 1: Deriving the Marginal Density

### 1. Proof via the Law of Total Probability
The latent discrete variable $C_i$ denotes the cluster assignment of observation $x_i$, taking values in $\{1, \dots, K\}$. By the Law of Total Probability, the marginal density $p(x_i)$ is computed by summing the joint probability density over all possible states of $C_i$:

$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k)$$

Using the definition of conditional probability, $p(x_i, C_i = k) = P(C_i = k) p(x_i \mid C_i = k)$. Substituting the prior probability $P(C_i = k) = \phi_k$ and the conditional multivariate Gaussian density $p(x_i \mid C_i = k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$, we obtain:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$

### 2. Physical Interpretation
This density is called a **Gaussian mixture density** because it is a convex linear combination (a weighted sum) of $K$ distinct component Gaussian distributions. Each individual Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ is scaled by its relative mixing proportion $\phi_k$, which governs its contribution to the overall global density.

## Part 2: Deriving the Posterior Cluster Probability

### 1. Applying Bayes' Rule
For a realized observation $X_i = x_i$, Bayes' rule dictates that the posterior probability of the latent variable belonging to cluster $k$ is:

$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

Substituting the expression for the marginal density $p(x_i)$ derived in Part 1 into the denominator yields:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) P(C_i = k)}{\sum_{j=1}^K P(X_i = x_i \mid C_i = j) P(C_i = j)}$$

### 2. GMM Parametric Form
Substituting our GMM assumptions ($P(C_i = k) = \phi_k$ and $p(x_i \mid C_i = k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$):

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$

### 3. Interpretation of Responsibilities
The quantity $\gamma_{ik}$ represents the **responsibility** that component $k$ takes for generating data point $x_i$. It acts as a posterior probability because it updates our *prior* belief ($\phi_k$) using the empirical *likelihood* ($\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$) of the observed coordinate under component $k$, naturally normalizing over all components such that $\sum_{k=1}^K \gamma_{ik} = 1$.

## Part 3: One-Hot Encoding of the Latent Cluster Variable

### 1. Conditional Expectation of Indicators
Let $Z_{ik}$ be a binary indicator variable where $Z_{ik} = 1$ if $C_i = k$ and $0$ otherwise. The conditional expectation of a binary random variable is explicitly equal to the probability of it taking the value 1:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = (1 \cdot P(Z_{ik} = 1 \mid X_i = x_i)) + (0 \cdot P(Z_{ik} = 0 \mid X_i = x_i))$$
$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

### 2. Vectorized Expectation
Extending this element-wise property to the full column vector $Z_i$:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \mathbb{E}[Z_{i2} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

### 3. Conclusion
Thus, the soft cluster assignment vector in a GMM is mathematically identical to the conditional expectation $\mathbb{E}[Z_i \mid X_i = x_i]$, capturing the probabilistic distribution of cluster membership over the entire simplex.

## Part 4: From Soft Assignment to Hard Clustering

### 1. Conceptual Distinction
* **Soft Clustering:** Provides a continuous distribution of cluster ownership via the vector $\mathbb{E}[Z_i \mid X_i = x_i]$. Point $x_i$ simultaneously belongs to *all* clusters with varying fractional weights ($\gamma_{ik}$). This preserves geometric ambiguity when a data point lies on a decision boundary.
* **Hard Clustering:** Enforces a mutually exclusive, deterministic assignment $\widehat{C}_i \in \{1, \dots, K\}$ via a Maximum A Posteriori (MAP) decision rule:

$$\widehat{C}_i = \arg\max_{1 \le k \le K} \gamma_{ik}$$

Hard clustering discards the underlying uncertainty, collapsing the continuous membership vector into a single discrete index.

## Part 5: Conditional Expectation of the Observation Given the Cluster

### 1. Component Center Derivation
Conditioned on knowing that $C_i = k$, the random variable $X_i$ follows a pure multivariate normal distribution: $X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$. By definition of the Gaussian distribution's first moment:

$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \mathscr{N}(x_i \mid \mu_k, \Sigma_k) dx_i = \mu_k$$

Hence, $\mu_k$ functions as the centroid or spatial center of cluster $k$.

### 2. Duality of the Expectations
* $\mathbb{E}[Z_i \mid X_i = x_i]$ maps an **observed point to cluster space**. It answers: *"Given this specific location in space, what is the probability profile of the hidden categories that generated it?"*
* $\mathbb{E}[X_i \mid C_i = k]$ maps a **hidden category to data space**. It answers: *"If we are looking exclusively inside cluster $k$, what is the expected average coordinate of its data points?"*

## Part 6: The Complete-Data Likelihood

### 1. Logarithmic Transformation
Assuming independent and identically distributed (i.i.d.) observations, the complete-data likelihood given both $x$ and the latent indicator matrix $z$ is:

$$p(x, z \mid \theta) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Applying the natural logarithm transforms the product operators into summations and distributes across the inner terms:

$$\ell_c = \log p(x, z \mid \theta) = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$
$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

### 2. Optimization Facility
If $z_{ik}$ were explicitly observed, this log-likelihood would decouple across components. Maximizing $\ell_c$ with respect to a specific cluster's parameters $(\mu_k, \Sigma_k)$ would only involve the subset of points assigned to it ($z_{ik}=1$), reducing the optimization task to standard, closed-form Maximum Likelihood Estimation (MLE) calculations for isolated Gaussians.

## Part 7: The EM Interpretation & The E-Step

### 1. Formulating the Q-Function
Because $Z_{ik}$ is unobserved, we cannot evaluate $\ell_c$ directly. The Expectation (E) step resolves this by computing the conditional expectation of $\ell_c$ with respect to the posterior distribution of the latent variables, given the current parameter estimates $\theta^{(t)}$:

$$Q(\theta, \theta^{(t)}) = \mathbb{E}_{Z \mid X, \theta^{(t)}}[\ell_c]$$
$$Q(\theta, \theta^{(t)}) = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i = x_i, \theta^{(t)}] \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Substituting $\mathbb{E}[Z_{ik} \mid X_i = x_i, \theta^{(t)}] = \gamma_{ik}$ yields:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

### 2. E-Step Synthesis
The E-step updates cluster membership probabilities dynamically. It takes the current global model state and computes a continuous distribution of ownership over every point, passing these fractional weights to the subsequent maximization step.

## Part 8: Parameter Updates (The M-Step)

To maximize $Q$ subject to the constraint $\sum_{k=1}^K \phi_k = 1$, we introduce the Lagrange multiplier $\lambda$:

$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k + \lambda \left(1 - \sum_{k=1}^K \phi_k\right)$$

Taking the partial derivative with respect to $\phi_k$ and setting it to zero:

$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda}$$

Summing both sides over all $K$ components to resolve $\lambda$:

$$\sum_{k=1}^K \phi_k = \frac{\sum_{i=1}^n \sum_{k=1}^K \gamma_{ik}}{\lambda} \implies 1 = \frac{\sum_{i=1}^n 1}{\lambda} \implies \lambda = n$$

Thus, defining the effective number of points assigned to cluster $k$ as $N_k = \sum_{i=1}^n \gamma_{ik}$, the updated mixing weight is:

$$\phi_k^{\text{new}} = \frac{N_k}{n}$$

For the spatial parameters $\mu_k$ and $\Sigma_k$, maximizing $Q$ isolates the standard Gaussian log-likelihood terms weighted by $\gamma_{ik}$. Differentiating with respect to $\mu_k$ and setting to zero yields:

$$\sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1}(x_i - \mu_k) = 0 \implies \mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$

Similarly, optimization for the dispersion structure yields:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

### Weighting Function of Responsibilities
The responsibility $\gamma_{ik}$ acts as a fractional membership weight. Instead of a point contributing fully to one cluster and zero to others, it shares its statistical influence proportionally across all components based on its posterior probability.

## Part 9: Structural Synthesis

Gaussian Mixture Model clustering is an iterative, optimization process based on the alternating conditional updates of latent cluster membership variables and geometric parameters.

* The mixture weight $\phi_k$ serves as the **prior probability** of cluster $k$, reflecting its expected global prevalence.
* The Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ quantifies spatial **compatibility**, measuring how likely it is that an observation emerged from component $k$'s geometry.
* Bayes' theorem harmonizes these elements to compute the **responsibility** $\gamma_{ik}$, the formal **posterior probability** of cluster membership given the data.
* Collectively, these terms comprise the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i]$, representing the coordinate's exact distribution over the latent simplex.
* The Maximization (M) step uses these posterior values as weights to update the cluster parameters, pulling the centers ($\mu_k$) and shapes ($\Sigma_k$) toward the regions where they take the highest statistical responsibility.

In summary, GMM is a robust probabilistic framework that replaces heuristic distance splitting with rigorous statistical inference rooted in conditional expectations.

## Part 9: Structural Synthesis

Gaussian Mixture Model clustering is an iterative, optimization process based on the alternating conditional updates of latent cluster membership variables and geometric parameters.

* The mixture weight $\phi_k$ serves as the **prior probability** of cluster $k$, reflecting its expected global prevalence.
* The Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ quantifies spatial **compatibility**, measuring how likely it is that an observation emerged from component $k$'s geometry.
* Bayes' theorem harmonizes these elements to compute the **responsibility** $\gamma_{ik}$, the formal **posterior probability** of cluster membership given the data.
* Collectively, these terms comprise the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i]$, representing the coordinate's exact distribution over the latent simplex.
* The Maximization (M) step uses these posterior values as weights to update the cluster parameters, pulling the centers ($\mu_k$) and shapes ($\Sigma_k$) toward the regions where they take the highest statistical responsibility.

In summary, GMM is a robust probabilistic framework that replaces heuristic distance splitting with rigorous statistical inference rooted in conditional expectations.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np


try:

    csv_files = glob.glob(os.path.join(path, "*.csv"))
    if csv_files:
        data_path = csv_files[0]
        print(f"Located CSV file: {data_path}")
    else:
        raise FileNotFoundError("No CSV file found in the downloaded directory.")
except NameError:
    print("Variable 'path' not found..")

    data_path = "CC GENERAL.csv"
df = pd.read_csv(data_path)
df.head()